# 정상과 가스만, 최대 다섯 입력으로 비교
후보 네 센서의 최근10샷 표준편차와 제품만 사용합니다. 같은 관측 구간을 학습/평가로 나누지 않는 평가를 중심으로, 한 센서·두 센서·네 센서 구성과 RG3 전용 모델을 비교합니다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold
from sklearn.metrics import average_precision_score, precision_score, recall_score

root=Path.cwd()
if not (root/'data').exists():
    root=root.parent
tables=root/'reports/tables'
original=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
gas=original.Reason.str.contains('가스',regex=False)
data=original.loc[original.PassOrFail.eq(0)|gas].reset_index(drop=True)
y=data.Reason.str.contains('가스',regex=False).astype(int)
assert len(data)==1290 and y.sum()==30
assert len(original)-len(data)==14
sensors={
 'barrel2':'Barrel_Temperature_2__past10_std',
 'hopper':'Hopper_Temperature__past10_std',
 'cushion':'Cushion_Position__past10_std',
 'barrel5':'Barrel_Temperature_5__past10_std'}
X=data[list(sensors.values())].copy()
X['product_RG3']=data['product'].eq('RG3').astype(int)
product=['product_RG3']
configs={
 'barrel2_only':product+[sensors['barrel2']],
 'hopper_only':product+[sensors['hopper']],
 'cushion_only':product+[sensors['cushion']],
 'barrel5_only':product+[sensors['barrel5']],
 'cn7_pair':product+[sensors['barrel2'],sensors['hopper']],
 'rg3_pair':product+[sensors['cushion'],sensors['barrel5']],
 'four_sensors_with_product':product+list(sensors.values()),
 'four_sensors_without_product':list(sensors.values())}
groups=data['comparison_block']
tie=np.random.RandomState(42).random_sample(len(data))
def model():
    return make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),
                         RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,
                                                class_weight='balanced_subsample',random_state=42,n_jobs=-1))
def evaluate(frame,target,group,cols,splitter,tie_values):
    probabilities=np.full(len(frame),np.nan)
    fold_rows=[]
    for train,test in splitter:
        assert not len(set(train)&set(test))
        fitted=model().fit(frame.iloc[train][cols],target.iloc[train])
        assert fitted[-1].n_features_in_==len(cols)
        probabilities[test]=fitted.predict_proba(frame.iloc[test][cols])[:,1]
        fold_rows.append({'block':str(group.iloc[test[0]]),'train_positive':int(target.iloc[train].sum()),
                          'test_positive':int(target.iloc[test].sum()),'test_n':len(test)})
    assert np.isfinite(probabilities).all()
    selected=np.zeros(len(frame),dtype=bool)
    for block in group.unique():
        idx=np.flatnonzero(group.eq(block).to_numpy())
        order=idx[np.lexsort((tie_values[idx],-probabilities[idx]))]
        selected[order[:max(1,int(np.ceil(len(idx)*.1)))]]=True
    predicted=probabilities>=.5
    stats={'features':len(cols),'shots':len(frame),'gas':int(target.sum()),
           'AP':average_precision_score(target,probabilities),
           'caught_0_5':int((predicted & target.astype(bool)).sum()),
           'false_positive_0_5':int((predicted & ~target.astype(bool)).sum()),
           'precision_0_5':precision_score(target,predicted,zero_division=0),
           'recall_0_5':recall_score(target,predicted,zero_division=0),
           'caught_top10pct':int((selected & target.astype(bool)).sum()),
           'selected_top10pct':int(selected.sum())}
    return stats,probabilities,selected,fold_rows

folds=list(LeaveOneGroupOut().split(X,y,groups))
for train,test in folds:
    assert set(groups.iloc[train]).isdisjoint(groups.iloc[test])
metrics=[]; predictions=[]; product_results=[]
for setting,cols in configs.items():
    stats,probability,selected,fold_rows=evaluate(X,y,groups,cols,folds,tie)
    stats['setting']=setting
    metrics.append(stats)
    for i in range(len(data)):
        predictions.append({'setting':setting,'product':data.iloc[i]['product'],
                            'TimeStamp':data.iloc[i]['TimeStamp'],'comparison_block':groups.iloc[i],
                            'Reason':data.iloc[i]['Reason'],'gas':int(y.iloc[i]),
                            'probability':probability[i],'selected_top10pct':int(selected[i])})
    for p in ['CN7','RG3']:
        idx=np.flatnonzero(data['product'].eq(p).to_numpy())
        product_results.append({'setting':setting,'product':p,'shots':len(idx),'gas':int(y.iloc[idx].sum()),
                                'AP':average_precision_score(y.iloc[idx],probability[idx]),
                                'caught_0_5':int(((probability[idx]>=.5)&y.iloc[idx].astype(bool)).sum()),
                                'caught_top10pct':int((selected[idx]&y.iloc[idx].astype(bool)).sum())})
    print(setting, 'inputs',len(cols),'AP',round(stats['AP'],4),'top10',stats['caught_top10pct'],'/30',flush=True)

# One optimistic reference; the same episode may be represented in both train and test.
reference,_,_,_=evaluate(X,y,groups,configs['four_sensors_with_product'],
                         list(StratifiedKFold(n_splits=5,shuffle=True,random_state=42).split(X,y)),tie)
reference['setting']='five_features_random_reference'
# RG3-specific models can leave a block out while retaining other positive blocks.
rg3_mask=data['product'].eq('RG3')
rg3=data.loc[rg3_mask].reset_index(drop=True)
rx=X.loc[rg3_mask].reset_index(drop=True)
ry=y.loc[rg3_mask].reset_index(drop=True)
rgroups=rg3['comparison_block']
rgfolds=list(LeaveOneGroupOut().split(rx,ry,rgroups))
separate=[]
for setting,cols in [('rg3_two_sensors',[sensors['cushion'],sensors['barrel5']]),
                     ('rg3_four_sensors',list(sensors.values()))]:
    stats,_,_,_=evaluate(rx,ry,rgroups,cols,rgfolds,np.random.RandomState(42).random_sample(len(rx)))
    stats['setting']=setting
    separate.append(stats)
cn7=data.loc[data['product'].eq('CN7')]
assert cn7.loc[cn7.Reason.str.contains('가스',regex=False),'comparison_block'].nunique()==1
metrics=pd.DataFrame(metrics)
metrics.to_csv(tables/'small_gas_feature_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(predictions).to_csv(tables/'small_gas_oof_predictions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(product_results).to_csv(tables/'small_gas_metrics_by_product.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(separate).to_csv(tables/'small_gas_rg3_separate_models.csv',index=False,encoding='utf-8-sig')
pd.DataFrame([reference]).to_csv(tables/'small_gas_random_reference.csv',index=False,encoding='utf-8-sig')
data[['product','TimeStamp','comparison_block','Reason']].join(X).assign(gas=y).to_csv(
    tables/'small_gas_feature_dataset.csv',index=False,encoding='utf-8-sig')

names={'barrel2_only':'제품 + 배럴2 흔들림','hopper_only':'제품 + 호퍼 흔들림',
       'cushion_only':'제품 + 쿠션 흔들림','barrel5_only':'제품 + 배럴5 흔들림',
       'cn7_pair':'제품 + 배럴2·호퍼 흔들림','rg3_pair':'제품 + 쿠션·배럴5 흔들림',
       'four_sensors_with_product':'제품 + 네 센서 흔들림',
       'four_sensors_without_product':'네 센서 흔들림만'}
chosen=metrics.loc[metrics.setting.eq('four_sensors_with_product')].iloc[0]
best=metrics.sort_values(['AP','features'],ascending=[False,True]).iloc[0]
lines=['# 적은 변수로 가스만 판별한 1차 결과','',
       '원본 라벨에서 정상 1,260 Shot과 가스 30 Shot만 사용했다. 가스·미성형 혼합 2 Shot은 가스 양성에 포함하고, 순수 미성형·초기허용불량 14 Shot은 제외했다. RH/LH는 같은 Shot으로 묶었다.',
       '',
       '입력은 최근 10개 관측 Shot의 표준편차 4개(배럴온도2, 호퍼온도, 쿠션 위치, 배럴온도5)와 제품 구분이며 최대 5개다. 결측값은 학습 데이터 중앙값으로 대체하고 결측 여부 변수를 추가하지 않았다. 직전 이력에는 비라벨 센서 기록도 연결했지만 라벨은 사용하지 않았다.',
       '',
       '모든 실험은 같은 Random Forest(200개 트리, 깊이8, 최소 말단3, 클래스 가중치, 시드42)를 사용했다. 원본 라벨 관측 구간 하나를 통째로 제외하는 8회 평가가 주요 결과다. 구간은 실제 Run ID가 아니다. 변수 선택과 비교에 현재 라벨을 사용했으므로 독립 최종 검증이 필요하다.',
       '',
       '## 주요 결과','',
       '다섯 변수 구성의 구간별 상위 약10% 검사에서 가스 {}/30 Shot을 발견했다. 고정 확률 0.5에서는 {} Shot을 발견하고 정상 {} Shot을 오탐했다. AP는 {:.3f}이다.'.format(
           int(chosen.caught_top10pct),int(chosen.caught_0_5),int(chosen.false_positive_0_5),chosen.AP),
       '',
       'AP가 가장 높은 탐색 구성은 {}({}개 입력), AP {:.3f}, 상위10% 발견 {}/30이다. 이 순위는 같은 평가 데이터로 고른 결과다.'.format(
           names[best.setting],int(best.features),best.AP,int(best.caught_top10pct)),
       '',
       '| 입력 구성 | 개수 | AP | 0.5 기준 발견/30 | 정상 오탐 | 구간별 상위10% 발견/30 |',
       '|---|---:|---:|---:|---:|---:|']
for r in metrics.itertuples(index=False):
    lines.append('| {} | {} | {:.3f} | {} | {} | {} |'.format(names[r.setting],r.features,r.AP,
                  r.caught_0_5,r.false_positive_0_5,r.caught_top10pct))
lines+=['','## 제품별로 확인한 다섯 변수 모델','',
        '| 제품 | 정상 Shot | 가스 Shot | AP | 상위10% 발견 |','|---|---:|---:|---:|---:|']
for r in pd.DataFrame(product_results).query("setting == 'four_sensors_with_product'").itertuples(index=False):
    lines.append('| {} | {} | {} | {:.3f} | {} |'.format(r.product,r.shots-r.gas,r.gas,r.AP,r.caught_top10pct))
lines+=['','## 제품별 모델','',
        'CN7의 가스 13 Shot은 모두 한 관측 구간에 있다. 그 구간을 평가용으로 빼면 CN7 학습 데이터에 가스가 0개여서, CN7 전용 모델의 새 구간 성능은 현재 데이터로 검증할 수 없다. 위 결합 모델의 CN7 결과는 RG3의 가스 사례를 함께 학습한 결과다.',
        '',
        '| RG3 전용 입력 | AP | 상위10% 발견/17 |','|---|---:|---:|']
for r in separate:
    lines.append('| {}개 센서 | {:.3f} | {} |'.format(r['features'],r['AP'],r['caught_top10pct']))
lines+=['','## 같은 사건을 공유하는 참고 시험','',
        '다섯 변수의 Shot 단위 임의 5분할에서는 AP {:.3f}, 상위10% 가스 발견 {}/30이었다. 같은 사건이 학습·평가에 함께 들어갈 수 있어 새 구간의 성능으로 해석하지 않는다.'.format(
            reference['AP'],reference['caught_top10pct']),
        '',
        '상위10% 순위는 동률일 때 고정 시드로 나누었다. 정상 기저율과 단순 검사 규모를 함께 고려해야 하며, 발견 숫자가 조금 높다는 이유만으로 유효성을 확정하지 않는다. 임계값0.5는 튜닝하지 않았다. Timestamp·센서 이력 제공 여부도 실제 테스트 전에 확인해야 한다.']
(root/'reports/small_gas_model_comparison.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('Five feature model:',chosen.to_dict())
print('RG3 separate:',separate)
print('Random reference:',reference)
print('All results saved.')
